# GenePT — Foundation Model Tutorial

**GenePT** — API-based GPT-3.5 gene embeddings (1536-dim), no local GPU required, gene-level (not cell-level)

| Property | Value |
|----------|-------|
| **Tasks** | embed |
| **Species** | human |
| **Gene IDs** | symbol |
| **GPU Required** | No (CPU OK) |
| **Min VRAM** | 0 GB |
| **Embedding Dim** | 1536 |
| **Repository** | [https://github.com/yiqunchen/GenePT](https://github.com/yiqunchen/GenePT) |


> **Note:** GenePT generates **gene-level** (not cell-level) embeddings using the OpenAI API. No local GPU required, but an OpenAI API key is needed.

This tutorial demonstrates how to use **GenePT** through the unified `ov.fm` API.

**Cite:** Zeng, Z. et al. (2024). OmicVerse: a framework for bridging and deepening insights across bulk and single-cell sequencing. *Nature Communications*, 15(1), 5983.

In [ ]:
import omicverse as ov
import scanpy as sc
import os
import warnings
warnings.filterwarnings('ignore')

ov.plot_set()

## Gene-level vs. cell-level embeddings

GenePT is fundamentally different from other models in `ov.fm`:

| Aspect | Cell-level models (scGPT, etc.) | GenePT |
|--------|--------------------------------|--------|
| Unit | One embedding per cell | One embedding per **gene** |
| Dimension | 200-1280 | **1536** |
| Source | Model inference | OpenAI API (GPT-3.5) |
| GPU | Required (most) | **Not required** |
| Cost | Compute | API cost |

Gene embeddings can be used for:

- Gene function similarity analysis
- Gene set enrichment with semantic matching
- Cell embeddings via weighted gene aggregation

## Step 1: Inspect Model Specification

Use `ov.fm.describe_model()` to get the full spec for GenePT.

In [ ]:
info = ov.fm.describe_model("genept")

print("=== Model Info ===")
print(f"Name: {info['model']['name']}")
print(f"Version: {info['model']['version']}")
print(f"Tasks: {info['model']['tasks']}")
print(f"Species: {info['model']['species']}")
print(f"Embedding dim: {info['model']['embedding_dim']}")
print(f"Differentiator: {info['model']['differentiator']}")

print("\n=== Input Contract ===")
print(f"Gene ID scheme: {info['input_contract']['gene_id_scheme']}")
print(f"Preprocessing: {info['input_contract']['preprocessing']}")

print("\n=== Output Contract ===")
print(f"Embedding key: {info['output_contract']['embedding_key']}")
print(f"Embedding dim: {info['output_contract']['embedding_dim']}")

## Step 2: Prepare Data

Load a dataset and save it for the `ov.fm` workflow. Most foundation models expect raw counts (non-negative values).

In [ ]:
# GenePT uses the OpenAI API to generate gene-level embeddings.
# No local GPU required, but you need an OpenAI API key:
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
#
# Note: GenePT produces GENE embeddings (1536-dim per gene),
# not CELL embeddings. Cell embeddings are derived by aggregating
# gene embeddings weighted by expression.

adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f'Dataset: {adata.n_obs} cells x {adata.n_vars} genes')

adata.write_h5ad('pbmc3k_genept.h5ad')


## Step 3: Profile Data & Validate Compatibility

Check whether your data is compatible with GenePT before running inference.

In [ ]:
profile = ov.fm.profile_data("pbmc3k_genept.h5ad")

print("=== Data Profile ===")
print(f"Species: {profile['species']}")
print(f"Gene scheme: {profile['gene_scheme']}")
print(f"Modality: {profile['modality']}")
print(f"Cells: {profile['n_cells']:,}")
print(f"Genes: {profile['n_genes']:,}")

# Validate compatibility
validation = ov.fm.preprocess_validate("pbmc3k_genept.h5ad", "genept", "embed")
print(f"\n=== Validation: {validation['status']} ===")
for d in validation.get("diagnostics", []):
    print(f"  [{d['severity']}] {d['message']}")
if validation.get("auto_fixes"):
    print("\nSuggested fixes:")
    for fix in validation["auto_fixes"]:
        print(f"  - {fix}")

## Step 4: Run GenePT Inference

Execute GenePT through `ov.fm.run()`. The function handles preprocessing, model loading, inference, and output writing.

In [ ]:
result = ov.fm.run(
    task="embed",
    model_name="genept",
    adata_path="pbmc3k_genept.h5ad",
    output_path="pbmc3k_genept_out.h5ad",
    device="auto",
)

if "error" in result:
    print(f"Error: {result['error']}")
    if "suggestion" in result:
        print(f"Suggestion: {result['suggestion']}")
else:
    print(f"Status: {result['status']}")
    print(f"Output keys: {result.get('output_keys', [])}")
    print(f"Cells processed: {result.get('n_cells', 0)}")

## Step 5: Visualize & Interpret Results

Load the output, compute UMAP from GenePT embeddings, and evaluate quality.

In [ ]:
if os.path.exists("pbmc3k_genept_out.h5ad"):
    adata_out = sc.read_h5ad("pbmc3k_genept_out.h5ad")
    emb_key = "X_genept"
    
    if emb_key in adata_out.obsm:
        print(f"Embedding shape: {adata_out.obsm[emb_key].shape}")
        
        # UMAP visualization
        sc.pp.neighbors(adata_out, use_rep=emb_key)
        sc.tl.umap(adata_out)
        sc.tl.leiden(adata_out, resolution=0.5)
        sc.pl.umap(adata_out, color=["leiden"],
                   title="GenePT Embedding (PBMC 3k)")
        
        # QA metrics
        interpretation = ov.fm.interpret_results("pbmc3k_genept_out.h5ad", task="embed")
        if "embeddings" in interpretation["metrics"]:
            for k, v in interpretation["metrics"]["embeddings"].items():
                print(f"\n{k}: dim={v['dim']}", end="")
                if "silhouette" in v:
                    print(f", silhouette={v['silhouette']:.4f}", end="")
                print()
    else:
        print(f"Embedding key {emb_key} not found.")
        print(f"Available keys: {list(adata_out.obsm.keys())}")
else:
    print("Output file not found — check model installation and adapter status.")
    print("See the Guide page for installation instructions.")

## Summary

| Step | Function | What it does |
|------|----------|-------------|
| 1 | `ov.fm.describe_model("genept")` | Inspect model spec and I/O contract |
| 2 | `sc.datasets.pbmc3k()` | Prepare input data |
| 3 | `ov.fm.profile_data()` + `preprocess_validate()` | Check compatibility |
| 4 | `ov.fm.run()` | Execute GenePT inference |
| 5 | `ov.fm.interpret_results()` | Evaluate embedding quality |

For the full model catalog, see `ov.fm.list_models()` or the [ov.fm API Overview](t_fm_guide.md).
For detailed GenePT specifications, see the [GenePT Guide](t_fm_genept_guide.md).